In [1]:
import pandas as pd
import numpy as np

In [2]:
lv = pd.read_csv("Data/Context/Latvia.csv")

In [3]:
lv = lv.drop(["Unnamed: 0"], axis=1)

In [4]:
lv_2007_2013 = lv.loc[7:13]

In [5]:
lv_2007_2013

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,WTDA,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
7,2007,0.72,0.43,0.34,0.95,0.83,48.0,0.958,0.922,59.2,...,3.97,NaN,5.0,16.0,88.154100,60.252809,60.0,81.115239,83.443983,NaN
8,2008,0.78,0.52,0.23,0.97,0.78,50.0,0.958,0.932,63.4,...,3.94,NaN,5.0,16.0,88.467773,60.736185,60.0,83.228626,83.443983,NaN
9,2009,0.79,0.62,0.20,0.92,0.85,45.0,0.958,0.933,66.8,...,3.67,NaN,5.0,15.5,89.002428,61.307267,60.0,83.043921,83.443983,NaN
10,2010,0.76,0.72,0.20,0.93,0.79,43.0,0.958,0.934,68.4,...,3.53,74.014114,5.0,15.5,89.196552,62.874544,60.0,82.613206,79.962144,NaN
11,2011,0.74,0.69,0.28,0.95,0.74,42.0,0.958,0.934,69.8,...,3.62,74.376122,5.0,15.5,89.163024,64.159569,60.0,81.896887,79.962144,NaN
12,2012,0.78,0.83,0.24,1.01,0.78,49.0,0.956,0.938,73.1,...,3.68,77.728796,4.0,15.5,91.495552,64.012406,60.0,82.315191,78.322799,NaN
13,2013,0.76,0.88,0.32,1.04,0.77,53.0,0.965,0.957,75.2,...,3.95,77.810472,4.0,15.5,91.532399,64.188060,60.0,84.229882,75.590559,NaN


In [6]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')

In [7]:
missing_values(lv_2007_2013)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 4 - 57.1 %
PR - 0 - 0.0 %
DI - 2 - 28.6 %
SOARS - 0 - 0.0 %
TOGP - 0 - 0.0 %
IPAB - 3 - 42.9 %
PTIP - 0 - 0.0 %
QOES - 0 - 0.0 %
AOLT - 0 - 0.0 %
FLTA - 0 - 0.0 %
EBOF - 0 - 0.0 %
FOWD - 0 - 0.0 %
QOMS - 0 - 0.0 %
CLER - 0 - 0.0 %
WTDA - 0 - 0.0 %
EDBS - 3 - 42.9 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 7 - 100.0 %


In [8]:
mode = lv_2007_2013['DI'].mode()[0]
lv_2007_2013['DI'].fillna(value=mode, inplace=True)

/var/folders/41/w_0mz3gn2vn0w0dz8wywr8_r0000gn/T/ipykernel_34545/1706926215.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lv_2007_2013['DI'].fillna(value=mode, inplace=True)


In [9]:
lv_2007_2013 = lv_2007_2013.drop(["ELFE", "CFI", "IPAB", "EDBS"], axis=1)

In [10]:
def fill_with_mean(df):
    for feature in df.columns:
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)

In [11]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [12]:
def handle_outliers(df, continuous_features):

    print("Original Shape:", df.shape)
    
    outlier_summary = {}
    
    for feature in continuous_features:
        if feature in df.columns:
            Q1 = df[feature].quantile(0.25)
            Q3 = df[feature].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (df[feature] < lower_bound) | (df[feature] > upper_bound)
            num_outliers = outlier_mask.sum()
            
            if num_outliers > 0:
                non_outlier_mean = df.loc[~outlier_mask, feature].mean()
                
                df.loc[outlier_mask, feature] = non_outlier_mean
                
                print(f"{feature}: Found {num_outliers} outliers, replaced with mean = {non_outlier_mean:.2f}")
            else:
                print(f"{feature}: No outliers found")
    
    print("\nFinal Shape:", df.shape)
    print(f"Total features processed: {len(continuous_features)}")
    
    return df

In [13]:
lv_2007_2013_no_o = handle_outliers(lv_2007_2013, continuous_data)

Original Shape: (7, 30)
DBP: No outliers found
DBD: No outliers found
SSB: No outliers found
DWCP: No outliers found
PT: No outliers found
EC: No outliers found
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 30)
Total features processed: 9


In [14]:
lv_2007_2013_no_o

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,QOMS,CLER,WTDA,DBP,DBD,SSB,DWCP,PMSI,PT,EC
7,2007,0.72,0.43,0.34,0.95,0.83,48.0,0.958,0.922,59.2,...,4.56,4.66,3.97,5.0,16.0,88.154100,60.252809,60.0,81.115239,83.443983
8,2008,0.78,0.52,0.23,0.97,0.78,50.0,0.958,0.932,63.4,...,4.36,4.30,3.94,5.0,16.0,88.467773,60.736185,60.0,83.228626,83.443983
9,2009,0.79,0.62,0.20,0.92,0.85,45.0,0.958,0.933,66.8,...,4.34,4.42,3.67,5.0,15.5,89.002428,61.307267,60.0,83.043921,83.443983
10,2010,0.76,0.72,0.20,0.93,0.79,43.0,0.958,0.934,68.4,...,4.17,4.29,3.53,5.0,15.5,89.196552,62.874544,60.0,82.613206,79.962144
11,2011,0.74,0.69,0.28,0.95,0.74,42.0,0.958,0.934,69.8,...,4.20,4.34,3.62,5.0,15.5,89.163024,64.159569,60.0,81.896887,79.962144
12,2012,0.78,0.83,0.24,1.01,0.78,49.0,0.956,0.938,73.1,...,4.23,4.37,3.68,4.0,15.5,91.495552,64.012406,60.0,82.315191,78.322799
13,2013,0.76,0.88,0.32,1.04,0.77,53.0,0.965,0.957,75.2,...,4.38,4.53,3.95,4.0,15.5,91.532399,64.188060,60.0,84.229882,75.590559


In [15]:
lv_2007_2013_no_o.to_csv('Data/Context/Latvia_2007_2013.csv')

In [16]:
lv_2007_2013_no_o = lv_2007_2013_no_o.drop(["TIME_PERIOD"], axis=1)

In [17]:
lv_2007_2013_l = lv_2007_2013_no_o.loc[13]

In [18]:
lv_2007_2013_l.to_csv('Data/Context/Latvia_2007_2013_last.csv')

In [19]:
lv_2014_2020 = lv.loc[14:20]

In [20]:

missing_values(lv_2014_2020)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 0 - 0.0 %
PR - 0 - 0.0 %
DI - 0 - 0.0 %
SOARS - 3 - 42.9 %
TOGP - 3 - 42.9 %
IPAB - 3 - 42.9 %
PTIP - 3 - 42.9 %
QOES - 3 - 42.9 %
AOLT - 3 - 42.9 %
FLTA - 3 - 42.9 %
EBOF - 3 - 42.9 %
FOWD - 3 - 42.9 %
QOMS - 3 - 42.9 %
CLER - 3 - 42.9 %
WTDA - 4 - 57.1 %
EDBS - 0 - 0.0 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 1 - 14.3 %


In [21]:
lv_2014_2020 = lv_2014_2020.drop(["SOARS", "TOGP", "IPAB", "PTIP", "QOES", "AOLT", "FLTA", "EBOF",
                                 "FOWD", "QOMS", "CLER", "WTDA"], axis=1)

In [22]:
fill_with_mean(lv_2014_2020)

In [23]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [24]:
lv_2014_2020_no_o = handle_outliers(lv_2014_2020, continuous_data)

Original Shape: (7, 22)
DBP: No outliers found
DBD: No outliers found
EDBS: No outliers found
SSB: No outliers found
DWCP: Found 2 outliers, replaced with mean = 73.43
PT: No outliers found
EC: Found 1 outliers, replaced with mean = 73.24
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 22)
Total features processed: 9


In [25]:
lv_2014_2020_no_o.to_csv('Data/Context/Latvia_2014_2020.csv')

In [26]:
lv_2014_2020_no_o = lv_2014_2020_no_o.drop(["TIME_PERIOD"], axis=1)

In [27]:
lv_2014_2020_no_o

,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,CFI,...,DI,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
14,0.87,0.95,0.41,1.17,0.85,55.0,0.967,0.954,75.8,44.8,...,7.48,77.907463,4.0,12.5,92.332633,73.432771,68.0,84.865341,75.590559,8.783333
15,0.77,1.05,0.43,1.07,0.85,55.0,0.967,0.954,79.2,45.5,...,7.37,78.042768,4.0,12.5,92.038243,73.432771,68.0,86.462455,73.237164,9.900000
16,0.93,0.97,0.40,1.07,0.84,57.0,0.964,0.954,79.8,44.3,...,7.31,79.158989,4.0,5.5,94.107515,73.392246,68.0,89.093857,71.655374,10.000000
17,0.90,0.87,0.50,1.14,0.80,58.0,0.950,0.954,80.1,44.6,...,7.25,80.624337,4.0,5.5,94.112781,73.387156,68.0,90.040534,71.655374,8.600000
18,0.92,1.00,0.30,1.13,0.79,58.0,0.963,0.963,83.6,43.2,...,7.38,80.017813,4.0,5.5,94.113949,73.413091,68.0,90.040534,73.507226,8.300000
19,0.98,1.07,0.48,1.18,0.86,56.0,0.972,0.963,86.1,43.2,...,7.49,80.386015,4.0,5.5,94.116764,73.456908,68.0,89.990298,73.507226,8.700000
20,0.92,0.84,0.69,1.18,0.87,57.0,0.961,0.963,88.9,41.1,...,7.24,80.340843,4.0,5.5,94.137989,73.514454,68.0,89.188135,73.507226,7.200000


In [28]:
lv_2014_2020_l = lv_2014_2020_no_o.loc[20]

In [29]:
lv_2014_2020_l

ROL      0.920000
GE       0.840000
COC      0.690000
RQ       1.180000
VAA      0.870000
CPI     57.000000
FOP      0.961000
JI       0.963000
IUI     88.900000
CFI     41.100000
PR      72.000000
DI       7.240000
EDBS    80.340843
DBP      4.000000
DBD      5.500000
SSB     94.137989
DWCP    73.514454
PMSI    68.000000
PT      89.188135
EC      73.507226
ELFE     7.200000
Name: 20, dtype: float64

In [30]:
lv_2014_2020_l.to_csv('Data/Context/Latvia_2014_2020_last.csv')